# Ensemble IKE Demo

This is a demo version of the ensemble IKE retrieval code.

- **MLP retriever**: frozen BERT encoder + trainable linear scoring head.
- **BERT retriever**: frozen BERT encoder + trainable linear scoring head.
- **LLM retriever**: listwise / pairwise LLM ranker using a text-generation model.
- **Feature-weighted ensemble**: learns per-query mixture weights over MLP/BERT/LLM.

By default, it runs on a tiny toy CounterFact-style dataset. To use real data, set `COUNTERFACT_PATH` and optionally `IKE_INDEX_PATH`.

## 0. Setup

Install dependencies if needed:
```bash
pip install torch transformers scikit-learn pandas numpy tqdm
```

For gated Hugging Face models such as Llama, authenticate outside the notebook:
```bash
huggingface-cli login
```

In [1]:
import json
import math
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    from transformers import AutoModelForCausalLM, AutoTokenizer, BertModel, BertTokenizer, pipeline
    HAS_TRANSFORMERS = True
except Exception as e:
    HAS_TRANSFORMERS = False
    torch = None
    nn = object
    BertModel = BertTokenizer = AutoTokenizer = AutoModelForCausalLM = pipeline = None
    print("Transformers/PyTorch not available. Install torch and transformers to run the real retrievers.")

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

SEED = 90
random.seed(SEED)
np.random.seed(SEED)
if HAS_TRANSFORMERS and torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (HAS_TRANSFORMERS and torch.cuda.is_available()) else "cpu"
print("Device:", DEVICE)

# Keep this False for a quick CPU-safe walkthrough. Set True to instantiate the actual
# BERT/MLP/LLM retrievers and evaluate them on the demo examples.
RUN_REAL_RETRIEVERS = False
RUN_LLM_RETRIEVER = False  # requires a generation model and more memory

BERT_MODEL_NAME = "bert-base-uncased"
LLM_RANKER_MODEL_NAME = os.environ.get("IKE_LLM_RANKER_MODEL", "meta-llama/Llama-3.2-3B-Instruct")

# Optional real data paths. Leave as None to use the toy data below.
COUNTERFACT_PATH: Optional[str] = None       # e.g., "counterfact.json"
IKE_INDEX_PATH: Optional[str] = None         # e.g., "corpus_idx.txt"
MAX_RECORDS = 40
NUM_CANDIDATES = 8

EPS = 1e-12


def softmax(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return x
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (ex.sum() + EPS)


def to_cpu_list(xs):
    out = []
    for x in xs:
        try:
            if hasattr(x, "detach"):
                out.append(float(x.detach().cpu().item()))
            else:
                out.append(float(x))
        except Exception:
            out.append(float(np.asarray(x)))
    return out

/Applications/anaconda3/envs/shap_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 1. CounterFact-style data helpers

In [2]:
def make_record(subject: str, prompt: str, target_new: str, target_true: Optional[str] = None) -> Dict[str, Any]:
    return {
        "requested_rewrite": {
            "subject": subject,
            "prompt": prompt,
            "target_new": {"str": target_new},
            "target_true": {"str": target_true or target_new},
            "relation_id": "demo_relation",
        },
        "paraphrase_prompts": [prompt.format(subject)],
        "neighborhood_prompts": [prompt.format(subject)],
    }


toy_records = [
    make_record("The Eiffel Tower", "{} is located in", "Paris"),
    make_record("The Colosseum", "{} is located in", "Rome"),
    make_record("The Statue of Liberty", "{} is located in", "New York City"),
    make_record("Mount Fuji", "{} is located in", "Japan"),
    make_record("The Taj Mahal", "{} is located in", "Agra"),
    make_record("The Louvre", "{} is located in", "Paris"),
    make_record("Big Ben", "{} is located in", "London"),
    make_record("The Great Wall", "{} is located in", "China"),
    make_record("The Sydney Opera House", "{} is located in", "Sydney"),
    make_record("Machu Picchu", "{} is located in", "Peru"),
    make_record("Harvard University", "{} is located in", "Cambridge"),
    make_record("Stanford University", "{} is located in", "California"),
]


def load_counterfact(path: Optional[str]) -> List[Dict[str, Any]]:
    if path is None:
        return toy_records
    with open(path, "r") as f:
        return json.load(f)


def fact_text(record: Dict[str, Any]) -> str:
    rr = record["requested_rewrite"]
    return rr["prompt"].format(rr["subject"]) + " " + rr["target_new"]["str"]


def load_ike_index(path: Optional[str]) -> Optional[List[List[int]]]:
    if path is None:
        return None
    rows = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append([int(x) for x in line.split()])
    return rows


def build_examples(records: List[Dict[str, Any]], num_candidates: int = 8, ike_index: Optional[List[List[int]]] = None, seed: int = 90):
    """
    Build retrieval examples. Each example contains a query, a gold fact, and candidate retains.
    If an IKE neighbor index is provided, candidates come from that index; otherwise we use
    random distractors so the demo runs immediately.
    """
    rng = random.Random(seed)
    facts = [fact_text(r) for r in records]
    examples = []
    for i, rec in enumerate(records):
        gold = fact_text(rec)
        if ike_index is not None and i < len(ike_index):
            neighbor_ids = [j for j in ike_index[i] if 0 <= j < len(records) and j != i]
            candidates = [gold] + [facts[j] for j in neighbor_ids[: max(0, num_candidates - 1)]]
        else:
            distractors = [f for j, f in enumerate(facts) if j != i]
            rng.shuffle(distractors)
            candidates = [gold] + distractors[: max(0, num_candidates - 1)]
        rng.shuffle(candidates)
        query_prompt = rec["requested_rewrite"]["prompt"].format(rec["requested_rewrite"]["subject"])
        examples.append({
            "idx": i,
            "query": query_prompt,
            "gold": gold,
            "candidates": candidates,
            "record": rec,
        })
    return examples


records = load_counterfact(COUNTERFACT_PATH)[:MAX_RECORDS]
ike_index = load_ike_index(IKE_INDEX_PATH)
examples = build_examples(records, NUM_CANDIDATES, ike_index, SEED)
print(f"Loaded {len(records)} records and built {len(examples)} retrieval examples")
pd.DataFrame([{k: v for k, v in ex.items() if k != "record"} for ex in examples[:3]])

Loaded 12 records and built 12 retrieval examples


,idx,query,gold,candidates
0,0,The Eiffel Tower is located in,The Eiffel Tower is located in Paris,"[Mount Fuji is located in Japan, Harvard Unive..."
1,1,The Colosseum is located in,The Colosseum is located in Rome,"[Harvard University is located in Cambridge, M..."
2,2,The Statue of Liberty is located in,The Statue of Liberty is located in New York City,"[Harvard University is located in Cambridge, T..."


## 2. Actual retriever implementations

All retrievers implement:

```python
retrieve_facts_list(query: str, candidates: list[str]) -> raw_scores, probs, ranked_candidates
```

This is the interface used by the original ensemble code.

In [3]:
if HAS_TRANSFORMERS:
    class MLPRetriever(nn.Module):
        """
        Frozen BERT representations followed by a trainable linear scoring head. In the full experiments, this head
        can be trained/calibrated; in this demo it can also be used untrained for a quick run.
        """
        def __init__(self, model_name: str = BERT_MODEL_NAME, tokenizer: Optional[Any] = None, device: str = DEVICE):
            super().__init__()
            self.device_name = device
            self.tokenizer = tokenizer or BertTokenizer.from_pretrained(model_name)
            self.bert = BertModel.from_pretrained(model_name)
            for param in self.bert.parameters():
                param.requires_grad = False
            self.linear = nn.Linear(self.bert.config.hidden_size, 1)
            self.to(device)

        def retrieve_facts(self, query: str, fact_dict: Dict[str, Any]):
            return self.retrieve_facts_list(query, list(fact_dict.keys()))

        def retrieve_facts_list(self, query: str, facts: List[str]):
            if not facts:
                return [], [], []
            inputs = self.tokenizer([query] + facts, padding=True, truncation=True, return_tensors="pt").to(self.device_name)
            with torch.no_grad():
                outputs = self.bert(**inputs)
                pooled_output = outputs.pooler_output
                scores = self.linear(pooled_output)
                candidate_scores = scores[1:].squeeze(-1)
                probs = torch.softmax(candidate_scores, dim=0)
                sorted_indices = torch.argsort(candidate_scores, descending=True)
            ranked_facts = [facts[i] for i in sorted_indices.detach().cpu().numpy().tolist()]
            return to_cpu_list(candidate_scores), to_cpu_list(probs), ranked_facts

        def query_pool(self, query: str):
            inputs = self.tokenizer([query], padding=True, truncation=True, return_tensors="pt").to(self.device_name)
            with torch.no_grad():
                outputs = self.bert(**inputs)
            return outputs.pooler_output.squeeze(0)

        def forward_logits(self, query: str, candidates: List[str]) -> torch.Tensor:
            raw_scores, _, _ = self.retrieve_facts_list(query, candidates)
            raw = torch.tensor(raw_scores, dtype=torch.float32, device=self.linear.weight.device).unsqueeze(-1)
            return raw.squeeze(-1)


    class BERTRetriever(nn.Module):
        """
        Actual BERT retriever structure from the research notebook: frozen BERT encoder
        plus a trainable linear layer that scores each candidate fact.
        """
        def __init__(self, model_name: str = BERT_MODEL_NAME, tokenizer: Optional[Any] = None, device: str = DEVICE):
            super().__init__()
            self.device_name = device
            self.tokenizer = tokenizer or BertTokenizer.from_pretrained(model_name)
            self.bert = BertModel.from_pretrained(model_name)
            for param in self.bert.parameters():
                param.requires_grad = False
            self.linear = nn.Linear(self.bert.config.hidden_size, 1)
            self.to(device)

        def retrieve_facts(self, query: str, fact_dict: Dict[str, Any]):
            return self.retrieve_facts_list(query, list(fact_dict.keys()))

        def retrieve_facts_list(self, query: str, facts: List[str]):
            if not facts:
                return [], [], []
            inputs = self.tokenizer([query] + facts, padding=True, truncation=True, return_tensors="pt").to(self.device_name)
            with torch.no_grad():
                outputs = self.bert(**inputs)
                pooled_output = outputs.pooler_output
                scores = self.linear(pooled_output)
                candidate_scores = scores[1:].squeeze(-1)
                probs = torch.softmax(candidate_scores, dim=0)
                sorted_indices = torch.argsort(candidate_scores, descending=True)
            ranked_facts = [facts[i] for i in sorted_indices.detach().cpu().numpy().tolist()]
            return to_cpu_list(candidate_scores), to_cpu_list(probs), ranked_facts


def clip_text(txt: str, limit: int = 280) -> str:
    txt = str(txt)
    return txt if len(txt) <= limit else txt[:limit] + " …"


def safe_json_extract(text: str):
    m = re.search(r"(\{.*\}|\[.*\])", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    return None


def make_chat_prompt(tokenizer, system_text: str, user_text: str):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return system_text + "\n\n" + user_text


def generate_text(model, tokenizer, prompt: str, temperature: float, top_p: float, max_new_tokens: int = 256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)


def aggregate_m_samples(score_lists):
    arrs = []
    for s in score_lists:
        s = np.asarray(s, dtype=np.float32)
        arrs.append((s - s.mean()) / (s.std() + 1e-8))
    return np.mean(arrs, axis=0)


def listwise_rank_fn(query, candidates, *, tokenizer, model, temperature=0.3, top_p=0.9, m=1):
    """
    LLM listwise ranker. It asks the LLM to score all candidates as JSON.
    Returns one raw score per candidate, where higher is better.
    """
    system = "You rank candidate facts for in-context editing. Return only valid JSON."
    numbered = "\n".join([f"{i}: {clip_text(c)}" for i, c in enumerate(candidates)])
    user = f"""
Query prompt: {query}

Candidate facts:
{numbered}

Return a JSON object mapping each candidate index to a relevance score from 0 to 100.
Example: {{"0": 92, "1": 17}}
""".strip()
    prompt = make_chat_prompt(tokenizer, system, user)
    samples = []
    for _ in range(m):
        text = generate_text(model, tokenizer, prompt, temperature, top_p)
        parsed = safe_json_extract(text) or {}
        scores = []
        for i in range(len(candidates)):
            try:
                scores.append(float(parsed.get(str(i), parsed.get(i, 0.0))))
            except Exception:
                scores.append(0.0)
        samples.append(scores)
    return aggregate_m_samples(samples)


def pairwise_bt_rank_fn(query, candidates, *, tokenizer, model, temperature=0.3, top_p=0.9, m=1):
    """
    Pairwise LLM ranker with a Bradley-Terry-style win count. This is used when the
    candidate list is larger than the listwise threshold.
    """
    wins = np.zeros(len(candidates), dtype=np.float64)
    system = "Choose which candidate fact is more relevant to the query. Return only A or B."
    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):
            user = f"""
Query prompt: {query}

A: {clip_text(candidates[i])}
B: {clip_text(candidates[j])}

Which candidate is more relevant, A or B?
""".strip()
            prompt = make_chat_prompt(tokenizer, system, user)
            votes_i = 0
            votes_j = 0
            for _ in range(m):
                text = generate_text(model, tokenizer, prompt, temperature, top_p, max_new_tokens=8).strip().upper()
                if text.startswith("A"):
                    votes_i += 1
                elif text.startswith("B"):
                    votes_j += 1
            if votes_i >= votes_j:
                wins[i] += 1
            else:
                wins[j] += 1
    return wins


@dataclass
class LLMRankerConfig:
    max_listwise_N: int = 12
    m_samples: int = 1
    temperature: float = 0.3
    top_p: float = 0.9
    seed: Optional[int] = 90


class LLMRanker:
    """
    It uses listwise ranking for small candidate sets and pairwise ranking for larger candidate sets.
    """
    def __init__(self, tokenizer_rank, model_rank, config: LLMRankerConfig,
                 listwise_rank_fn=listwise_rank_fn, pairwise_bt_rank_fn=pairwise_bt_rank_fn):
        self.tok = tokenizer_rank
        self.model = model_rank
        self.cfg = config
        self.listwise_rank_fn = listwise_rank_fn
        self.pairwise_bt_rank_fn = pairwise_bt_rank_fn
        if self.cfg.seed is not None:
            random.seed(self.cfg.seed)
            np.random.seed(self.cfg.seed)

    def normalize_to_probs(self, raw_scores: np.ndarray) -> np.ndarray:
        raw_scores = np.asarray(raw_scores, dtype=np.float64)
        if raw_scores.size == 0:
            return raw_scores
        xs = (raw_scores - raw_scores.mean()) / (raw_scores.std() + 1e-8)
        return softmax(xs)

    def retrieve_facts_list(self, query: str, candidates: List[str]):
        if not candidates:
            return [], [], []
        if len(candidates) <= self.cfg.max_listwise_N:
            raw = self.listwise_rank_fn(
                query, candidates, tokenizer=self.tok, model=self.model,
                temperature=self.cfg.temperature, top_p=self.cfg.top_p, m=self.cfg.m_samples,
            )
        else:
            raw = self.pairwise_bt_rank_fn(
                query, candidates, tokenizer=self.tok, model=self.model,
                temperature=self.cfg.temperature, top_p=self.cfg.top_p, m=self.cfg.m_samples,
            )
        raw = np.asarray(raw, dtype=np.float32)
        probs = self.normalize_to_probs(raw)
        order = np.argsort(-raw)
        ranked = [candidates[i] for i in order]
        return raw.tolist(), probs.tolist(), ranked

# Optional lightweight retriever for quick smoke tests when RUN_REAL_RETRIEVERS=False.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class TfidfSmokeTestRetriever:
    def retrieve_facts_list(self, query: str, candidates: List[str]):
        if not candidates:
            return [], [], []
        vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
        X = vectorizer.fit_transform([query] + candidates)
        scores = cosine_similarity(X[0], X[1:]).ravel()
        probs = softmax(scores)
        order = np.argsort(-scores)
        return scores.tolist(), probs.tolist(), [candidates[i] for i in order]

## 3. Instantiate retrievers

In [4]:
def load_llm_ranker(model_name: str = LLM_RANKER_MODEL_NAME):
    if not HAS_TRANSFORMERS:
        raise RuntimeError("Install torch and transformers before loading the LLM ranker.")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True,
    )
    if DEVICE == "cpu":
        model.to("cpu")
    return LLMRanker(tokenizer, model, LLMRankerConfig())


retrievers = {}
if RUN_REAL_RETRIEVERS:
    if not HAS_TRANSFORMERS:
        raise RuntimeError("RUN_REAL_RETRIEVERS=True requires torch and transformers.")
    shared_bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
    retrievers["mlp"] = MLPRetriever(BERT_MODEL_NAME, tokenizer=shared_bert_tokenizer, device=DEVICE)
    retrievers["bert"] = BERTRetriever(BERT_MODEL_NAME, tokenizer=shared_bert_tokenizer, device=DEVICE)
    if RUN_LLM_RETRIEVER:
        retrievers["llm"] = load_llm_ranker(LLM_RANKER_MODEL_NAME)
else:
    # Quick runnable placeholder. The actual BERT/MLP/LLM classes above are still included
    # so users can switch RUN_REAL_RETRIEVERS=True when they have the dependencies/models.
    retrievers["smoke_test_tfidf"] = TfidfSmokeTestRetriever()

print("Active retrievers:", list(retrievers))

Active retrievers: ['smoke_test_tfidf']


## 4. Evaluation helpers

In [5]:
def evaluate_retriever(retriever, examples: List[Dict[str, Any]], name: str):
    rows = []
    for ex in tqdm(examples, desc=f"{name} eval"):
        raw_scores, probs, ranked = retriever.retrieve_facts_list(ex["query"], ex["candidates"])
        pred = ranked[0] if ranked else None
        rows.append({
            "idx": ex["idx"],
            "model": name,
            "query": ex["query"],
            "gold": ex["gold"],
            "pred": pred,
            "correct": int(pred == ex["gold"]),
            "retains": ranked,
            "probs": probs,
            "scores": raw_scores,
            "candidates": ex["candidates"],
        })
    return rows


def summarize_accuracy(results_by_name: Dict[str, List[Dict[str, Any]]]):
    rows = []
    for name, res in results_by_name.items():
        acc = np.mean([r["correct"] for r in res]) if res else 0.0
        rows.append({"retriever": name, "top1_accuracy": acc, "n": len(res)})
    return pd.DataFrame(rows).sort_values("top1_accuracy", ascending=False)


results_by_name = {name: evaluate_retriever(ret, examples, name) for name, ret in retrievers.items()}
summarize_accuracy(results_by_name)

smoke_test_tfidf eval: 100%|██████████| 12/12 [00:00<00:00, 556.51it/s]


,retriever,top1_accuracy,n
0,smoke_test_tfidf,1.0,12


## 5. Feature-conditioned ensemble over MLP / BERT / LLM outputs

This mirrors the original ensemble idea: extract query-level features from each retriever's
probability distribution, learn which retriever tends to be correct, and combine their scores.

For a real run, use at least two active retrievers by setting:
```python
RUN_REAL_RETRIEVERS = True
RUN_LLM_RETRIEVER = True  # optional, heavier
```

In [6]:
def norm_entropy(probs):
    p = np.asarray(to_cpu_list(probs), dtype=np.float64)
    if p.size == 0:
        return 0.0
    p = p / (p.sum() + EPS)
    h = -(p * np.log(p + EPS)).sum()
    return float(h / np.log(len(p) + EPS))


def top1_prob(probs):
    p = np.asarray(to_cpu_list(probs), dtype=np.float64)
    return 0.0 if p.size == 0 else float(np.max(p))


def top1_margin(probs):
    p = np.asarray(to_cpu_list(probs), dtype=np.float64)
    if p.size < 2:
        return 1.0
    top2 = np.partition(p, -2)[-2:]
    return float(top2.max() - top2.min())


def char_ngram_score(query: str, retains: List[str], top_k: int = 8, n: int = 3):
    def grams(s):
        s = re.sub(r"\s+", " ", s.lower())
        return {s[i:i+n] for i in range(max(0, len(s)-n+1))}
    qg = grams(query)
    best = 0.0
    for r in retains[:top_k]:
        rg = grams(r)
        best = max(best, len(qg & rg) / max(1, len(qg | rg)))
    return float(best)


def extract_query_features(rec_by_name: Dict[str, Dict[str, Any]]):
    # Basic text features from the query
    any_rec = next(iter(rec_by_name.values()))
    q = any_rec["query"]
    num_chars = len(q)
    num_tokens = len(q.split())
    avg_tok_len = num_chars / max(1, num_tokens)
    frac_caps = sum(1 for c in q if c.isupper()) / max(1, num_chars)
    frac_punct = sum(1 for c in q if (not c.isalnum() and not c.isspace())) / max(1, num_chars)

    feats = [num_chars, num_tokens, avg_tok_len, frac_caps, frac_punct]
    for name in sorted(rec_by_name):
        rec = rec_by_name[name]
        probs = rec["probs"]
        feats.extend([
            norm_entropy(probs),
            top1_margin(probs),
            top1_prob(probs),
            char_ngram_score(q, rec["retains"]),
        ])
    return np.asarray(feats, dtype=np.float64)


def build_weight_training_table(results_by_name: Dict[str, List[Dict[str, Any]]]):
    names = sorted(results_by_name)
    n = len(next(iter(results_by_name.values())))
    X, Y = [], []
    for i in range(n):
        recs = {name: results_by_name[name][i] for name in names}
        X.append(extract_query_features(recs))
        y = np.array([float(recs[name]["correct"] == 1) for name in names], dtype=np.float64)
        if y.sum() == 0:
            y[:] = 1.0 / len(y)
        else:
            y = y / y.sum()
        Y.append(y)
    return np.vstack(X), np.vstack(Y), names


class GBWeightModel:
    def __init__(self, max_depth=3, learning_rate=0.05, max_iter=100):
        self.reg = MultiOutputRegressor(
            HistGradientBoostingRegressor(
                loss="squared_error",
                max_depth=max_depth,
                learning_rate=learning_rate,
                max_iter=max_iter,
                random_state=SEED,
            )
        )
        self.mu = None
        self.sd = None

    def fit(self, X, Y):
        self.mu = X.mean(axis=0)
        self.sd = X.std(axis=0) + EPS
        self.reg.fit((X - self.mu) / self.sd, Y)
        return self

    def predict_weights(self, x, names: List[str]):
        raw = np.asarray(self.reg.predict(((x - self.mu) / self.sd).reshape(1, -1))[0], dtype=np.float64)
        w = softmax(raw)
        return {name: float(val) for name, val in zip(names, w)}


class GatedFeatureWeightedRetriever:
    def __init__(self, weight_model: GBWeightModel, retrievers: Dict[str, Any], gate_thr: float = 0.15):
        self.wm = weight_model
        self.retrievers = {k: retrievers[k] for k in sorted(retrievers)}
        self.names = sorted(retrievers)
        self.gate_thr = gate_thr

    def retrieve_facts_list(self, query: str, candidates: List[str]):
        recs = {}
        aligned_probs = {}
        for name, retriever in self.retrievers.items():
            raw, probs, ranked = retriever.retrieve_facts_list(query, candidates)
            # Align probabilities to the original candidate order.
            raw_arr = np.asarray(raw, dtype=np.float64)
            if raw_arr.size != len(candidates):
                raw_arr = np.zeros(len(candidates), dtype=np.float64)
            p = softmax(raw_arr)
            aligned_probs[name] = p
            recs[name] = {"query": query, "probs": p.tolist(), "retains": ranked}

        x = extract_query_features(recs)
        weights = self.wm.predict_weights(x, self.names)
        ordered = sorted(weights.items(), key=lambda kv: kv[1], reverse=True)
        best_name, best_w = ordered[0]
        second_w = ordered[1][1] if len(ordered) > 1 else 0.0

        if best_w - second_w >= self.gate_thr:
            mix = aligned_probs[best_name]
        else:
            mix = sum(weights[name] * aligned_probs[name] for name in self.names)

        order = np.argsort(-mix)
        ranked = [candidates[i] for i in order]
        return mix.tolist(), mix.tolist(), ranked


if len(results_by_name) >= 2:
    Xw, Yw, names = build_weight_training_table(results_by_name)
    wm = GBWeightModel(max_iter=100).fit(Xw, Yw)
    ens = GatedFeatureWeightedRetriever(wm, retrievers, gate_thr=0.15)
    ens_rows = evaluate_retriever(ens, examples, "ensemble_gated")
    results_with_ens = {**results_by_name, "ensemble_gated": ens_rows}
    display(summarize_accuracy(results_with_ens))
else:
    print("Enable at least two retrievers to train/evaluate the ensemble.")

Enable at least two retrievers to train/evaluate the ensemble.


## 6. Inspect retrieved examples

In [7]:
for name, rows in results_by_name.items():
    print("\n===", name, "===")
    for row in rows[:3]:
        print("Query:", row["query"])
        print("Gold: ", row["gold"])
        print("Pred: ", row["pred"])
        print("Correct:", row["correct"])
        print("Top candidates:")
        for cand in row["retains"][:3]:
            print("  -", cand)
        print("-" * 80)


=== smoke_test_tfidf ===
Query: The Eiffel Tower is located in
Gold:  The Eiffel Tower is located in Paris
Pred:  The Eiffel Tower is located in Paris
Correct: 1
Top candidates:
  - The Eiffel Tower is located in Paris
  - The Colosseum is located in Rome
  - The Great Wall is located in China
--------------------------------------------------------------------------------
Query: The Colosseum is located in
Gold:  The Colosseum is located in Rome
Pred:  The Colosseum is located in Rome
Correct: 1
Top candidates:
  - The Colosseum is located in Rome
  - The Louvre is located in Paris
  - The Taj Mahal is located in Agra
--------------------------------------------------------------------------------
Query: The Statue of Liberty is located in
Gold:  The Statue of Liberty is located in New York City
Pred:  The Statue of Liberty is located in New York City
Correct: 1
Top candidates:
  - The Statue of Liberty is located in New York City
  - The Louvre is located in Paris
  - The Taj Mahal 

## 7. Suggested path from demo to full release

1. Set `COUNTERFACT_PATH` to the real CounterFact JSON file.
2. (Optional) Set `IKE_INDEX_PATH` to the location of a precomputed IKE retrieval index. The IKE method retrieves demonstrations using a nearest-neighbor index built over the training examples. If no precomputed index is available, the notebook will construct the index from the provided training data.
3. Set `RUN_REAL_RETRIEVERS = True` to instantiate MLP and BERT retrievers.
4. Set `RUN_LLM_RETRIEVER = True` only when you have model access and enough memory.
5. Add trained checkpoint loading for the MLP/BERT scoring heads if you want to reproduce
paper-level results instead of a structural demo.
6. Add the full answer-generation evaluation after retrieval correctness has been verified.